Connected to reactive-agent (3.11.x) (Python 3.11.-1)

 # Short-Term Memory (STM)

 **One job:** give LangGraph a persistent connection pool so it can checkpoint
 graph state to PostgreSQL after every node execution.

 LangGraph's `AsyncPostgresSaver` needs a native psycopg3 connection.
 It can't use the SQLAlchemy/asyncpg engine from `database.py` —
 different drivers, incompatible at the protocol level.
 So two pools exist in this project: asyncpg for SQLAlchemy queries,
 psycopg3 for the checkpointer. Same database, different connections.


<div align="center">
  <img src="image/shtm_1.png" width="800" padding="30"/>
</div>


In [ ]:
from urllib.parse import urlsplit, urlunsplit
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from app.core.config import get_settings
from psycopg_pool import AsyncConnectionPool

settings = get_settings()
_pool: AsyncConnectionPool | None = None
_checkpointer: AsyncPostgresSaver | None = None

 ## `_normalize_psycopg_dsn`

 One `DATABASE_URL` in `.env`, two drivers that want different formats.
 SQLAlchemy wants `postgresql+asyncpg://`, psycopg3 wants `postgresql://`.
 This function strips the `+asyncpg` part and normalizes `postgres://`
 to `postgresql://` — psycopg3 rejects the short form.

In [ ]:
def _normalize_psycopg_dsn(dsn: str) -> str:
    parsed = urlsplit(dsn)
    scheme = parsed.scheme.split("+")[0]  # strip +asyncpg, +psycopg, etc.
    if scheme == "postgres":
        scheme = "postgresql"             # psycopg3 rejects "postgres://"
    return urlunsplit((scheme, parsed.netloc, parsed.path, parsed.query, parsed.fragment))

 ## `init_checkpointer`

 Called once in the FastAPI lifespan at startup.

 - `open=False` then `await _pool.open()` — the pool can't open in `__init__`
   because it's async. Create synchronously, open explicitly after.
 - `autocommit=True` — required by `AsyncPostgresSaver`.
 - `prepare_threshold=0` — disables server-side prepared statements,
   which cause issues with PgBouncer in transaction pooling mode.
 - `await _checkpointer.setup()` — creates the LangGraph checkpoint tables
   if they don't exist. Must run before the first graph invocation.

In [ ]:
async def init_checkpointer() -> None:
    global _pool, _checkpointer
    _pool = AsyncConnectionPool(
        conninfo=_normalize_psycopg_dsn(str(settings.database_url)),
        min_size=2,
        max_size=10,
        kwargs={"autocommit": True, "prepare_threshold": 0},
        open=False,
    )
    await _pool.open()
    _checkpointer = AsyncPostgresSaver(conn=_pool)
    await _checkpointer.setup()

 ## `get_checkpointer`

 Synchronous getter — no `await` needed.
 Raises immediately if called before `init_checkpointer()` instead of
 returning `None` silently. Fail fast is the right behavior here.

In [ ]:
def get_checkpointer() -> AsyncPostgresSaver:
    if _checkpointer is None:
        raise RuntimeError("Checkpointer not initialized — call init_checkpointer() at startup")
    return _checkpointer

 ## `close_checkpointer`

 Called in the FastAPI lifespan on shutdown — closes the psycopg3 pool cleanly.

In [ ]:
async def close_checkpointer() -> None:
    global _pool
    if _pool:
        await _pool.close()

 ## `get_thread_config`

 Produces the `config` dict LangGraph needs to isolate conversation history.

 - Same `thread_id` → graph resumes from last checkpoint (same conversation)
 - Different `thread_id` → fresh start
 - Namespaced as `user_id:session_id` to prevent collisions between users

In [ ]:
def get_thread_config(session_id: str, user_id: str) -> dict:
    return {
        "configurable": {
            "thread_id": f"{user_id}:{session_id}",
            "user_id": user_id,
        }
    }


<div align="center">
  <img src="image/shtm_2.png" width="800" padding="10"/>
</div>
